# Números — Redes Neurais e KNN

Este notebook compara **Rede Neural (MLP)** e **KNN** em dois cenários:

1. modelos treinados com o `dataset digits` do scikit-learn;
2. modelos treinados com as 100 imagens do pacote MNIST fornecido.

A comparação principal usa o **mesmo conjunto de teste MNIST-100**, separado antes do treinamento. Assim, avaliamos diretamente o efeito de treinar com exemplos do dataset adicionado.

## Antes de executar

Coloque o arquivo `dataset_mnist_100.zip` na mesma pasta deste notebook e execute as células em ordem. O notebook extrairá automaticamente as imagens para `imagens_usuario`. Se essa pasta já existir e contiver imagens, ela será usada sem alterações.

Também continuam aceitos arquivos próprios com nomes como `7_meu_digito.png`, `digito_7_ana.jpg`, `classe_7.png` e `ana_7.jpg`. Use apenas um dígito isolado (0–9) no nome.

In [2]:
# Se alguma biblioteca estiver ausente, descomente e execute a linha abaixo.
# %pip install numpy pandas pillow matplotlib seaborn scikit-learn joblib

from pathlib import Path, PurePosixPath
import re
import shutil
import warnings
import zipfile

import joblib
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from PIL import Image
from sklearn.datasets import load_digits
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
from sklearn.model_selection import train_test_split
from sklearn.neighbors import KNeighborsClassifier
from sklearn.neural_network import MLPClassifier
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

SEMENTE = 42
PASTA_IMAGENS = Path('imagens_usuario')
ARQUIVO_DATASET = Path('dataset_mnist_100.zip')
EXTENSOES = {'.png', '.jpg', '.jpeg', '.bmp', '.webp'}
sns.set_theme(style='whitegrid', context='notebook')

## 1. Leitura dos rótulos e pré-processamento

As imagens adicionadas podem ter tamanhos, fundos e margens diferentes. A função abaixo converte tudo para tons de cinza, detecta fundo claro, recorta o traço, centraliza o conteúdo e produz uma imagem 8×8 compatível com `digits`.

In [3]:
def extrair_rotulo(caminho: Path) -> int:
    nome = caminho.stem.lower()
    padroes = [
        r'^(?P<rotulo>[0-9])(?:[_\-]|$)',
        r'(?:digito|dígito|classe|numero|número|num)[_\- ]*(?P<rotulo>[0-9])(?:[_\- ]|$)',
    ]
    for padrao in padroes:
        achado = re.search(padrao, nome)
        if achado:
            return int(achado.group('rotulo'))

    digitos_isolados = re.findall(r'(?<![0-9])([0-9])(?![0-9])', nome)
    if len(digitos_isolados) == 1:
        return int(digitos_isolados[0])
    raise ValueError('nome sem um rótulo único e claro entre 0 e 9')


def preprocessar_imagem(caminho: Path, tamanho=8, margem=1) -> np.ndarray:
    with Image.open(caminho) as img:
        arr = np.asarray(img.convert('L'), dtype=np.float32) / 255.0

    # O resultado deve ter fundo preto e traço claro, como sklearn digits.
    borda = np.concatenate([arr[0], arr[-1], arr[:, 0], arr[:, -1]])
    if np.median(borda) > 0.5:
        arr = 1.0 - arr

    # Remove pequenas variações do fundo e encontra a região do dígito.
    arr = np.clip((arr - 0.08) / 0.92, 0.0, 1.0)
    mascara = arr > 0.12
    if not mascara.any():
        raise ValueError('nenhum traço foi detectado')

    ys, xs = np.where(mascara)
    recorte = arr[ys.min():ys.max() + 1, xs.min():xs.max() + 1]
    lado_util = tamanho - 2 * margem
    escala = min(lado_util / recorte.shape[1], lado_util / recorte.shape[0])
    nova_largura = max(1, round(recorte.shape[1] * escala))
    nova_altura = max(1, round(recorte.shape[0] * escala))
    metodo = getattr(Image, 'Resampling', Image).LANCZOS
    pequeno = Image.fromarray(np.uint8(recorte * 255)).resize(
        (nova_largura, nova_altura), metodo
    )

    canvas = np.zeros((tamanho, tamanho), dtype=np.float32)
    y0 = (tamanho - nova_altura) // 2
    x0 = (tamanho - nova_largura) // 2
    canvas[y0:y0 + nova_altura, x0:x0 + nova_largura] = (
        np.asarray(pequeno, dtype=np.float32) / 255.0
    )
    return canvas


def preparar_dataset(pasta: Path, arquivo_zip: Path):
    existentes = [
        p for p in pasta.rglob('*')
        if p.is_file() and p.suffix.lower() in EXTENSOES
    ] if pasta.exists() else []
    if existentes:
        print(f'Usando {len(existentes)} imagens existentes em {pasta.resolve()}')
        return

    if not arquivo_zip.exists():
        raise FileNotFoundError(
            f'Não encontrei {arquivo_zip.name}. Coloque o ZIP na mesma pasta do notebook.'
        )

    membros_validos = []
    with zipfile.ZipFile(arquivo_zip) as pacote:
        for info in pacote.infolist():
            partes = PurePosixPath(info.filename).parts
            if info.is_dir() or not partes or partes[0] != pasta.name:
                continue
            if '..' in partes:
                raise ValueError('O ZIP contém um caminho inseguro.')
            destino = Path(*partes)
            if destino.suffix.lower() in EXTENSOES:
                membros_validos.append((info, destino))

        if not membros_validos:
            raise ValueError(
                f'O arquivo {arquivo_zip.name} não contém imagens em {pasta.name}/.'
            )

        for info, destino in membros_validos:
            destino.parent.mkdir(parents=True, exist_ok=True)
            with pacote.open(info) as origem, destino.open('wb') as saida:
                shutil.copyfileobj(origem, saida)

    print(f'{len(membros_validos)} imagens extraídas de {arquivo_zip.name}.')


def carregar_imagens_usuario(pasta: Path):
    if not pasta.exists():
        raise FileNotFoundError(
            f'A pasta {pasta.resolve()} não existe. Crie-a ao lado do notebook.'
        )

    arquivos = sorted(
        p for p in pasta.rglob('*') if p.is_file() and p.suffix.lower() in EXTENSOES
    )
    imagens, rotulos, caminhos, erros = [], [], [], []
    for caminho in arquivos:
        try:
            rotulo = extrair_rotulo(caminho)
            imagem = preprocessar_imagem(caminho)
            imagens.append(imagem)
            rotulos.append(rotulo)
            caminhos.append(caminho)
        except Exception as exc:
            erros.append((str(caminho), str(exc)))

    if erros:
        print('Arquivos ignorados:')
        display(pd.DataFrame(erros, columns=['arquivo', 'motivo']))
    if not imagens:
        raise ValueError('Nenhuma imagem válida foi encontrada.')

    return np.stack(imagens), np.asarray(rotulos), caminhos


In [4]:
preparar_dataset(PASTA_IMAGENS, ARQUIVO_DATASET)
imagens_externas, y_externo, caminhos_externos = carregar_imagens_usuario(PASTA_IMAGENS)
X_externo = imagens_externas.reshape(len(imagens_externas), -1)

contagem = pd.Series(y_externo).value_counts().reindex(range(10), fill_value=0).sort_index()
display(contagem.rename('quantidade').to_frame().T)

classes_problematicas = contagem[contagem < 10]
if len(classes_problematicas):
    warnings.warn(
        'Há classes com menos de 10 imagens: ' +
        ', '.join(f'{classe} ({qtd})' for classe, qtd in classes_problematicas.items())
    )

ax = contagem.plot.bar(figsize=(9, 3), color='#4C78A8', rot=0)
ax.axhline(10, color='#E45756', linestyle='--', label='mínimo recomendado')
ax.set(title='Quantidade de imagens por classe', xlabel='Dígito', ylabel='Imagens')
ax.legend()
plt.show()

100 imagens extraídas de dataset_mnist_100.zip.


,0,1,2,3,4,5,6,7,8,9
quantidade,10,10,10,10,10,10,10,10,10,10


<Figure size 900x300 with 1 Axes>

In [5]:
# Confira visualmente se os rótulos e o pré-processamento ficaram corretos.
rng = np.random.default_rng(SEMENTE)
indices = rng.choice(len(imagens_externas), size=min(20, len(imagens_externas)), replace=False)
fig, eixos = plt.subplots(4, 5, figsize=(9, 7))
for eixo in eixos.flat:
    eixo.axis('off')
for eixo, indice in zip(eixos.flat, indices):
    eixo.imshow(imagens_externas[indice], cmap='gray', vmin=0, vmax=1)
    eixo.set_title(f'rótulo: {y_externo[indice]}')
    eixo.axis('off')
plt.suptitle('Amostra das imagens após o pré-processamento')
plt.tight_layout()
plt.show()

<Figure size 900x700 with 20 Axes>

## 2. Preparação dos conjuntos de treino e teste

A separação estratificada preserva aproximadamente a proporção das classes. As imagens do teste MNIST-100 não participam do treinamento dos modelos ajustados com o dataset adicionado.

In [6]:
if set(contagem[contagem > 0].index) != set(range(10)):
    faltantes = sorted(set(range(10)) - set(contagem[contagem > 0].index))
    raise ValueError(f'Faltam imagens das classes: {faltantes}')
if contagem.min() < 2:
    raise ValueError('Cada classe precisa de pelo menos 2 imagens para separar treino e teste.')

Xg_treino, Xg_teste, yg_treino, yg_teste = train_test_split(
    X_externo, y_externo, test_size=0.25, random_state=SEMENTE, stratify=y_externo
)

digits = load_digits()
X_digits = digits.data.astype(np.float32) / 16.0
y_digits = digits.target
Xd_treino, Xd_teste, yd_treino, yd_teste = train_test_split(
    X_digits, y_digits, test_size=0.25, random_state=SEMENTE, stratify=y_digits
)

pd.DataFrame({
    'conjunto': ['digits', 'MNIST-100'],
    'treino': [len(Xd_treino), len(Xg_treino)],
    'teste': [len(Xd_teste), len(Xg_teste)],
})

,conjunto,treino,teste
0,digits,1347,450
1,MNIST-100,75,25


## 3. Treinamento de Rede Neural e KNN

Usamos as mesmas configurações de cada algoritmo nos dois conjuntos. A padronização fica dentro do `Pipeline`, sendo ajustada somente com os dados de treino.

In [7]:
def criar_modelos():
    return {
        'Rede Neural': Pipeline([
            ('escala', StandardScaler()),
            ('modelo', MLPClassifier(
                hidden_layer_sizes=(64, 32),
                activation='relu',
                solver='adam',
                alpha=1e-4,
                max_iter=1500,
                random_state=SEMENTE,
                early_stopping=False,
            )),
        ]),
        'KNN': Pipeline([
            ('escala', StandardScaler()),
            ('modelo', KNeighborsClassifier(n_neighbors=3, weights='distance')),
        ]),
    }

modelos_digits = criar_modelos()
modelos_externos = criar_modelos()

for modelo in modelos_digits.values():
    modelo.fit(Xd_treino, yd_treino)
for modelo in modelos_externos.values():
    modelo.fit(Xg_treino, yg_treino)

print('Treinamento concluído.')

Treinamento concluído.


## 4. Comparação das acurácias

A tabela completa cruza origem do treino e origem do teste. Para responder à pergunta central, compare as linhas no teste `MNIST-100`: todos os modelos recebem exatamente as mesmas imagens de teste.

In [8]:
resultados = []
predicoes_externas = {}
for origem, modelos in [('digits', modelos_digits), ('MNIST-100', modelos_externos)]:
    for nome, modelo in modelos.items():
        for teste, X, y in [
            ('digits', Xd_teste, yd_teste),
            ('MNIST-100', Xg_teste, yg_teste),
        ]:
            pred = modelo.predict(X)
            resultados.append({
                'treino': origem,
                'modelo': nome,
                'teste': teste,
                'acurácia': accuracy_score(y, pred),
            })
            if teste == 'MNIST-100':
                predicoes_externas[(origem, nome)] = pred

df_resultados = pd.DataFrame(resultados)
tabela = df_resultados.pivot(index=['modelo', 'treino'], columns='teste', values='acurácia')
display(tabela.map(lambda valor: f'{valor:.2%}'))

plt.figure(figsize=(10, 4))
dados_grafico = df_resultados.copy()
dados_grafico['configuração'] = dados_grafico['modelo'] + ' | treino: ' + dados_grafico['treino']
ax = sns.barplot(data=dados_grafico, x='configuração', y='acurácia', hue='teste')
ax.set_ylim(0, 1.05)
ax.set(xlabel='', ylabel='Acurácia', title='Acurácia por origem de treino e teste')
ax.tick_params(axis='x', rotation=20)
ax.yaxis.set_major_formatter(lambda x, pos: f'{x:.0%}')
plt.tight_layout()
plt.show()

teste                 MNIST-100  digits
modelo      treino                     
KNN         MNIST-100    68.00%  25.11%
            digits       32.00%  96.67%
Rede Neural MNIST-100    60.00%  27.33%
            digits       28.00%  96.67%

<Figure size 1000x400 with 1 Axes>

### Comparação principal: reconhecimento das imagens MNIST-100

Uma diferença positiva significa que treinar com imagens MNIST-100 melhorou a acurácia naquele algoritmo. Como o conjunto é pequeno, a conclusão deve mencionar também o número de imagens de teste.

In [9]:
comparacao_externa = (
    df_resultados.query("teste == 'MNIST-100'")
    .pivot(index='modelo', columns='treino', values='acurácia')
    .rename(columns={'digits': 'treinado em digits', 'MNIST-100': 'treinado no MNIST-100'})
)
comparacao_externa['diferença (MNIST-100 - digits)'] = (
    comparacao_externa['treinado no MNIST-100'] - comparacao_externa['treinado em digits']
)
comparacao_formatada = comparacao_externa.copy()
comparacao_formatada['treinado em digits'] = comparacao_formatada['treinado em digits'].map('{:.2%}'.format)
comparacao_formatada['treinado no MNIST-100'] = comparacao_formatada['treinado no MNIST-100'].map('{:.2%}'.format)
comparacao_formatada['diferença (MNIST-100 - digits)'] = comparacao_formatada['diferença (MNIST-100 - digits)'].map('{:+.2%}'.format)
display(comparacao_formatada)

treino,treinado no MNIST-100,treinado em digits,diferença (MNIST-100 - digits)
modelo,,,
KNN,68.00%,32.00%,+36.00%
Rede Neural,60.00%,28.00%,+32.00%


## 5. Matrizes de confusão no teste MNIST-100

As matrizes mostram quais dígitos são confundidos. Isso é mais informativo do que olhar apenas a acurácia total.

In [10]:
fig, eixos = plt.subplots(2, 2, figsize=(14, 11))
for eixo, ((origem, nome), pred) in zip(eixos.flat, predicoes_externas.items()):
    matriz = confusion_matrix(yg_teste, pred, labels=range(10))
    sns.heatmap(matriz, annot=True, fmt='d', cmap='Blues', cbar=False, ax=eixo)
    eixo.set_title(f'{nome} — treino: {origem} | teste: MNIST-100')
    eixo.set_xlabel('Predito')
    eixo.set_ylabel('Verdadeiro')
plt.tight_layout()
plt.show()

# Relatórios detalhados para os modelos treinados com MNIST-100.
for nome in modelos_externos:
    print(f'\n{nome} — treino e teste MNIST-100')
    print(classification_report(
        yg_teste, predicoes_externas[('MNIST-100', nome)], labels=range(10), zero_division=0
    ))

<Figure size 1400x1100 with 4 Axes>


Rede Neural — treino e teste MNIST-100
              precision    recall  f1-score   support

           0       1.00      1.00      1.00         2
           1       0.33      0.50      0.40         2
           2       0.50      0.33      0.40         3
           3       0.00      0.00      0.00         3
           4       0.67      0.67      0.67         3
           5       0.50      0.50      0.50         2
           6       1.00      0.67      0.80         3
           7       0.67      1.00      0.80         2
           8       0.25      0.50      0.33         2
           9       0.75      1.00      0.86         3

    accuracy                           0.60        25
   macro avg       0.57      0.62      0.58        25
weighted avg       0.57      0.60      0.57        25


KNN — treino e teste MNIST-100
              precision    recall  f1-score   support

           0       0.50      1.00      0.67         2
           1       1.00      1.00      1.00         2
      

## 6. Exemplos de erros

Observe se os erros têm traços muito finos, dígitos deslocados, baixa iluminação, cortes ou estilos parecidos com outra classe.

In [11]:
for nome in modelos_externos:
    pred = predicoes_externas[('MNIST-100', nome)]
    erros = np.flatnonzero(pred != yg_teste)
    print(f'{nome}: {len(erros)} erro(s) em {len(yg_teste)} imagens de teste')
    if len(erros) == 0:
        continue
    fig, eixos = plt.subplots(1, min(10, len(erros)), figsize=(15, 2.2))
    eixos = np.atleast_1d(eixos)
    for eixo, indice in zip(eixos, erros[:10]):
        eixo.imshow(Xg_teste[indice].reshape(8, 8), cmap='gray', vmin=0, vmax=1)
        eixo.set_title(f'real {yg_teste[indice]}\npred {pred[indice]}')
        eixo.axis('off')
    plt.suptitle(f'Erros — {nome} treinado com MNIST-100')
    plt.tight_layout()
    plt.show()

Rede Neural: 10 erro(s) em 25 imagens de teste


<Figure size 1500x220 with 10 Axes>

KNN: 8 erro(s) em 25 imagens de teste


<Figure size 1500x220 with 8 Axes>

## 7. Texto-base para a conclusão

A célula abaixo insere os números medidos. Revise o texto e acrescente observações feitas nas matrizes de confusão e nos exemplos de erro.

In [12]:
linhas = []
for modelo, linha in comparacao_externa.iterrows():
    acc_digits = linha['treinado em digits']
    acc_externo = linha['treinado no MNIST-100']
    diferenca = linha['diferença (MNIST-100 - digits)']
    direcao = 'maior' if diferenca > 0 else 'menor' if diferenca < 0 else 'igual'
    linhas.append(
        f'No {modelo}, a acurácia no teste MNIST-100 foi {acc_digits:.1%} quando o modelo '
        f'foi treinado com digits e {acc_externo:.1%} quando foi treinado com as imagens '
        f'MNIST-100, uma diferença de {abs(diferenca):.1%} ({direcao}).'
    )

melhor = comparacao_externa['treinado no MNIST-100'].idxmax()
texto = (
    'Foram usadas ' + str(len(y_externo)) + ' imagens MNIST-100, das quais ' +
    str(len(yg_teste)) + ' ficaram reservadas para teste. ' + ' '.join(linhas) + ' ' +
    f'Entre os modelos treinados com MNIST-100, o melhor resultado foi obtido por {melhor}. ' +
    'As diferenças podem ser explicadas pelo estilo dos traços, pelo pré-processamento, ' +
    'pela quantidade e diversidade de exemplos e pela diferença de distribuição entre ' +
    'o dataset digits e o MNIST-100. Como o conjunto MNIST-100 é pequeno, uma ' +
    'nova divisão treino/teste pode alterar os percentuais; portanto, os resultados devem ' +
    'ser interpretados junto com as matrizes de confusão.'
)
print(texto)

Foram usadas 100 imagens MNIST-100, das quais 25 ficaram reservadas para teste. No KNN, a acurácia no teste MNIST-100 foi 32.0% quando o modelo foi treinado com digits e 68.0% quando foi treinado com as imagens MNIST-100, uma diferença de 36.0% (maior). No Rede Neural, a acurácia no teste MNIST-100 foi 28.0% quando o modelo foi treinado com digits e 60.0% quando foi treinado com as imagens MNIST-100, uma diferença de 32.0% (maior). Entre os modelos treinados com MNIST-100, o melhor resultado foi obtido por KNN. As diferenças podem ser explicadas pelo estilo dos traços, pelo pré-processamento, pela quantidade e diversidade de exemplos e pela diferença de distribuição entre o dataset digits e o MNIST-100. Como o conjunto MNIST-100 é pequeno, uma nova divisão treino/teste pode alterar os percentuais; portanto, os resultados devem ser interpretados junto com as matrizes de confusão.
